In [27]:
import pandas as pd
import requests
import json
import os
url = 'https://catalogodatos.cnmc.es/dataset/8a2461de-07b9-4bbc-9e56-db06270f888a/resource/e63f5db2-0433-4b72-8647-e5a4c74f1876/download/ds_14042_1.json'
response = requests.get(url)

# FORZAMOS LA CODIFICACIÓN PARA IGNORAR EL BOM
response.encoding = 'utf-8-sig'
data = response.json()
df = pd.DataFrame(data)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 14691 entries, 0 to 14690
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   fecha_precio                 14691 non-null  str    
 1   provincia                    14691 non-null  str    
 2   producto                     14691 non-null  str    
 3   promedio_de_pai_diario_cubo  14691 non-null  float64
 4   promedio_de_pvp_diario_cubo  14691 non-null  float64
dtypes: float64(2), str(3)
memory usage: 574.0 KB


In [28]:
# Convertimos a formato datetime la columna 'fecha_precio'
df['fecha_precio'] = pd.to_datetime(df['fecha_precio'])

# Obtenemos última fecha del df
ultimo_dia = df['fecha_precio'].max()

# Fecha  inicio de la guerra 
fecha_28_feb = pd.Timestamp(year=ultimo_dia.year, month=2, day=28)

# Join
df_28feb = df[df['fecha_precio'] == fecha_28_feb].copy()
df_ultimo = df[df['fecha_precio'] == ultimo_dia].copy()

resultado = pd.merge(
    df_28feb, 
    df_ultimo[['provincia', 'producto', 'promedio_de_pai_diario_cubo', 'promedio_de_pvp_diario_cubo']], 
    on=['provincia', 'producto'], 
    how='inner', 
    suffixes=('_28feb', '_ultimo')
)

# Incrementos
resultado['incremento_pai'] = (resultado['promedio_de_pai_diario_cubo_ultimo'] - resultado['promedio_de_pai_diario_cubo_28feb']) / resultado['promedio_de_pai_diario_cubo_28feb'] * 100
resultado['incremento_pvp'] = (resultado['promedio_de_pvp_diario_cubo_ultimo'] - resultado['promedio_de_pvp_diario_cubo_28feb']) / resultado['promedio_de_pvp_diario_cubo_28feb'] * 100
resultado['incremento_pai'] = resultado['incremento_pai'].round(1)
resultado['incremento_pvp'] = resultado['incremento_pvp'].round(1)
price_increase_iran_war_2026 = resultado[['provincia', 'producto', 'incremento_pvp']]
# print(resultado.head())

In [29]:
url = 'https://unpkg.com/es-atlas@0.6.0/es/provinces.json' # DATOS OBTENIDOS DE https://observablehq.com/@martgnz/mapa-de-espana-con-topojson-y-es-atlas
response = requests.get(url)
topo_data = response.json()

In [ ]:
# PROVINCIAS CON NOMBRES DISTINTOS EN LOS 2 ARCHIVOS
mapeo_provincias = {
    'Alicante/Alacant': 'Alacant/Alicante',
    'Palmas, Las': 'Las palmas',
    'Balears, Illes': 'Illes Balears',
    'Valencia/València': 'València/Valencia',
    'Castellón/Castelló': 'Castelló/Castellón',
    'Rioja, La': 'La Rioja',
    'Araba/Álava': 'Araba/Álava',
    'Coruña, A': 'A Coruña'
}



price_increase_iran_war_2026['provincia'] = price_increase_iran_war_2026['provincia'].replace(mapeo_provincias)

In [ ]:
# Pivotemos los productos
df_pivot = price_increase_iran_war_2026.pivot(
    index='provincia', 
    columns='producto', 
    values='incremento_pvp'
).fillna(0) # Si falta un dato, ponemos 0

# Convertir a diccionario: { "ALBACETE": {"Gasolina 95": 0.18, "Gasóleo A": 0.35}, ... }
df_pivot.index = df_pivot.index.str.upper()
datos_dict = df_pivot.to_dict(orient='index')

# Añadimos TopoJSON
provincias = topo_data['objects']['provinces']['geometries']
for p in provincias:
    # nombre_mapa = p['properties']['name'].split('/')[-1].strip().upper()
    nombre_mapa = p['properties']['name'].strip().upper()
    p['properties']['datos_gasolina'] = datos_dict.get(nombre_mapa, {})

In [ ]:
# 1. Definir la ruta y el nombre del archivo
relative_path = os.path.join('..', 'src', 'assets', 'data')
file_name = 'cartogram_price_increase_iran_war_2026.json'
full_path = os.path.join(relative_path, file_name)

with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(
        topo_data,            
        f, 
        ensure_ascii=False,    # Permite ver acentos como 'ó' en lugar de '\u00f3'
        indent=4,              # Crea saltos de línea y sangrías (legible)
        sort_keys=True        
    )

✅ Archivo guardado con éxito en: ..\src\assets\data\price_increase_iran_war_2026.json
